# Caching and Persistence

This notebook covers caching strategies for improving Spark performance.

## Learning Objectives

- Understand when to cache data
- Learn different storage levels
- Monitor cache usage
- Avoid common caching pitfalls

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("Caching-and-Persistence") \
    .getOrCreate()

## 1. Why Cache?

Caching stores data in memory to avoid recomputation. Use it when:

1. Data is used multiple times
2. Data is expensive to compute
3. Data fits in available memory

In [ ]:
# Create a DataFrame that's expensive to compute
df = spark.range(1, 1000000).repartition(10)

# Add some transformations
transformed_df = df \
    .withColumn("squared", col("id") * col("id")) \
    .withColumn("cubed", col("id") * col("id") * col("id")) \
    .filter(col("id") % 2 == 0)

print("DataFrame created with expensive transformations")

## 2. cache() vs persist()

- `cache()`: Shortcut for `persist(StorageLevel.MEMORY_AND_DISK)`
- `persist()`: Allows specifying storage level

In [ ]:
# Using cache()
transformed_df.cache()

# First action - triggers computation and caching
print("First count (computes and caches):")
transformed_df.count()

In [ ]:
# Second action - uses cached data
print("Second count (uses cache):")
transformed_df.count()

In [ ]:
# Check if cached
print(f"Is cached: {transformed_df.is_cached}")

## 3. Storage Levels

Different storage levels offer trade-offs between memory usage and CPU.

In [ ]:
# Available storage levels
print("Available Storage Levels:")
print("MEMORY_ONLY: Store in memory only (spills if not enough)")
print("MEMORY_AND_DISK: Store in memory, spill to disk if needed")
print("DISK_ONLY: Store on disk only")
print("MEMORY_ONLY_2: Same as MEMORY_ONLY but replicate to 2 nodes")
print("MEMORY_AND_DISK_2: Same as MEMORY_AND_DISK but replicate to 2 nodes")

In [ ]:
# Using persist with specific storage level
df2 = spark.range(1, 100000).repartition(5)

# Persist to memory only
df2.persist(StorageLevel.MEMORY_ONLY)

df2.count()  # Trigger caching
print("DataFrame persisted with MEMORY_ONLY")

## 4. Unpersisting

Always unpersist when you're done to free memory.

In [ ]:
# Unpersist the DataFrames
transformed_df.unpersist()
df2.unpersist()

print("DataFrames unpersisted")

## 5. Monitoring Cache in Spark UI

The Spark UI shows cached data in the Storage tab.

- Access at: http://localhost:4040
- Click "Storage" tab
- See cached RDDs/DataFrames and their memory usage

In [ ]:
# Create and cache a DataFrame to see in UI
try:
    users = spark.read.parquet("/opt/spark/data/users/small")
    users.cache()
    users.count()  # Trigger caching
    
    print("Users DataFrame cached!")
    print("Check the Storage tab in Spark UI: http://localhost:4040")
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 6. When to Cache

### Good candidates for caching:
- Data used in multiple actions
- Data used in iterative algorithms (ML)
- Data that's expensive to compute
- Small to medium datasets that fit in memory

### Avoid caching:
- Data used only once
- Very large datasets
- Data that's cheap to recompute

In [ ]:
# Example: Good use of caching
try:
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    
    # Cache filtered data used multiple times
    large_orders = orders.filter(col("total_amount") > 500).cache()
    
    # Multiple operations on same filtered data
    print("Count of large orders:", large_orders.count())
    print("Average amount:", large_orders.agg({"total_amount": "avg"}).collect()[0][0])
    print("Max amount:", large_orders.agg({"total_amount": "max"}).collect()[0][0])
    
    large_orders.unpersist()
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 7. Exercises

In [ ]:
# Exercise 1: Cache the products DataFrame and use it in 3 different queries
# Your code here:


In [ ]:
# Exercise 2: Compare execution time with and without caching
# Hint: Use time module to measure
# Your code here:


In [ ]:
# Clean up
spark.catalog.clearCache()
spark.stop()